In [77]:
import os
import sys
import random , pathlib
import subprocess
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import numpy as np
import pandas as pd
import re

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import faiss

In [78]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
set_seed()

In [79]:
class SentenceTransformersBackend():
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")
def build_embedding_backend(
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu"):

    backend = SentenceTransformersBackend(model_name=model_name, device=device)
    print("Используем полноценные dense embeddings.")
    print("Бэкэнд:", backend.backend_name)
    return backend
   
embedder = build_embedding_backend(device=DEVICE)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3014.50it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [80]:
def chunk_text(text: str, chunk_size: int = 22, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks



def build_chunks(
    data_path:str,
    chunk_size: int = 22,
    overlap: int = 5,
    extra_data=None
) -> pd.DataFrame:
    rows = []
    c= 0
    paths= [data_path+doc for doc in os.listdir(data_path)]
    if extra_data!=None:
        paths+=[extra_data+doc for doc in os.listdir(extra_data)]
    for doc in paths:
        #print(doc)
        text = open(doc,"r",encoding='utf-8').read()
        chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": str(c),
                    "title": doc,
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split()),
                }
            )
        c+=1

    return rows
def build_chunks_dataframe(
    data_path:str,
    chunk_size: int = 22,
    overlap: int = 5,
    extra_data=None
) -> pd.DataFrame:


    return pd.DataFrame(build_chunks(data_path,chunk_size,overlap,extra_data=extra_data))


chunks_df = build_chunks_dataframe("data/", chunk_size=18, overlap=5)

print("Количество чанков:", len(chunks_df))
display(chunks_df.head(10))

Количество чанков: 513


,doc_id,title,chunk_id,chunk_text,n_words
0,0,data/Software Defined Radio — как это работает...,0,"Привет, Хабр. Продолжая цикл статей про радио,...",18
1,0,data/Software Defined Radio — как это работает...,1,в этой области — Software Defined Radio. Я не ...,18
2,0,data/Software Defined Radio — как это работает...,2,"на русский, поэтому оставим так, да и термин S...",18
3,0,data/Software Defined Radio — как это работает...,3,и радиолюбительских кругах. За последние 100 л...,18
4,0,data/Software Defined Radio — как это работает...,4,"тогдашний инженер вообще понял бы, как это раб...",18
5,0,data/Software Defined Radio — как это работает...,5,История Идея software defined radio базируется...,18
6,0,data/Software Defined Radio — как это работает...,6,радиоприемника в компьютер. Ширина обрабатывае...,18
7,0,data/Software Defined Radio — как это работает...,7,до 50МГц (сверхбыстрый АЦП с передачей сигнала...,18
8,0,data/Software Defined Radio — как это работает...,8,"сигнала — все то, что «обычный» радиоприемник ...",18
9,0,data/Software Defined Radio — как это работает...,9,"в «железе» — в SDR делается на компьютере, мат...",18


In [81]:
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError
@dataclass
class RetrievalArtifacts:
    backend_name: str
    backend: EmbeddingBackend
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    index: object


def build_retriever(
    docs: str,
    chunk_size: int = 40,
    overlap: int = 10,
    device: str = "cpu",
    extra_data=None
) -> RetrievalArtifacts:
    chunks = build_chunks(docs, chunk_size=chunk_size, overlap=overlap,extra_data=extra_data)
    chunks_df = pd.DataFrame(chunks)

    backend = embedder
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist())

    

    index = faiss.IndexFlatIP(chunk_vectors.shape[1])
    index.add(chunk_vectors)

    return RetrievalArtifacts(
        backend_name=backend.backend_name,
        backend=backend,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        index=index,
    )


def search_chunks(
    query: str,
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype("float32")
    scores, indices = artifacts.index.search(query_vector, top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = artifacts.chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "score": float(score),
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": chunk_row["chunk_id"],
                "chunk_text": chunk_row["chunk_text"],
            }
        )
    return pd.DataFrame(rows)


def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered


def evaluate_query(
    query: str,
    relevant_doc_ids: List[str],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> Dict[str, object]:
    result_df = search_chunks(query, artifacts=artifacts, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break

    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }


def evaluate_benchmark(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            artifacts=artifacts,
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "expected_source": ", ".join(row["relevant_doc_ids"]),
                "retrieved_sources": ", ".join(metrics["predicted_doc_ids"]),
                f"hit_at_k": metrics["hit"],
                f"recall@{top_k}": metrics["recall"],
                f"MRR@{top_k}": metrics["mrr"],
                "rank_of_first_relevant": metrics["first_relevant_rank"],
            }
        )
    return pd.DataFrame(rows)

In [82]:
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)
chunk_embeddings.shape[1]

384

In [83]:
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None


        
        self._faiss_index = faiss.IndexFlatIP(dim)  # type: ignore[name-defined]
        self.backend_name = "FAISS IndexFlatIP"
        

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        
        self._faiss_index.add(vectors)
        
        

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        
        
        


search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

print("Индекс построен.")
print("Бэкэнд индекса:", search_index.backend_name)

def search_similar_chunks(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vectors = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vectors, top_k=top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": int(chunk_row["chunk_id"]),
                "score": round(float(score), 4),
                "chunk_text": chunk_row["chunk_text"],
            }
        )

    return pd.DataFrame(rows)

faiss_query = "блоки Throttle Multiply Cons для чего нужны??"
faiss_results_df = search_similar_chunks(faiss_query, top_k=5)

display(Markdown(f"**Запрос:** {faiss_query}"))
display(faiss_results_df)

Индекс построен.
Бэкэнд индекса: FAISS IndexFlatIP


**Запрос:** блоки Throttle Multiply Cons для чего нужны??

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,2,data/Software Defined Radio — как это работает...,47,0.5782,"saved:"", 2*d_info.ret) block += 1 if block > m..."
1,2,2,data/Software Defined Radio — как это работает...,44,0.4917,"blocks to save block, max_blocks = 0, 10 block..."
2,3,3,data/Хакинг бытовых устройств программно-опред...,200,0.4910,забудем задать нашей кнопке имя. Далее вставля...
3,4,3,data/Хакинг бытовых устройств программно-опред...,180,0.4814,"выбирать нужные блоки, перетаскивать их и соед..."
4,5,3,data/Хакинг бытовых устройств программно-опред...,188,0.4529,"в ходе предыдущей атаки. Далее в другом блоке,..."


In [84]:
baseline_chunk_size = 28
baseline_overlap = 8

artifacts = build_retriever(
    "data/",
    chunk_size=baseline_chunk_size,
    overlap=baseline_overlap,
    device=DEVICE,
)

print("Используемый backend:", artifacts.backend_name)
print("Количество чанков:", len(artifacts.chunks_df))
display(artifacts.chunks_df.head())

Используемый backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Количество чанков: 334


,doc_id,title,chunk_id,chunk_text,n_words
0,0,data/Software Defined Radio — как это работает...,0,"Привет, Хабр. Продолжая цикл статей про радио,...",28
1,0,data/Software Defined Radio — как это работает...,1,Я не знаю адекватного перевода термина на русс...,28
2,0,data/Software Defined Radio — как это работает...,2,радиолюбительских кругах. За последние 100 лет...,28
3,0,data/Software Defined Radio — как это работает...,3,Мы все же попробуем разобраться. История Идея ...,28
4,0,data/Software Defined Radio — как это работает...,4,компьютер. Ширина обрабатываемой полосы может ...,28


In [85]:
benchmark_queries: List[Dict[str, object]] = [
    {
        "query_id": "q01",
        "query": "Что такое SDR и зачем он нужен?",
        "relevant_doc_ids": ["0"],
    },
    {
        "query_id": "q02",
        "query": "Зачем нужен GNU Radio?",
        "relevant_doc_ids": ["1"],
    },
    {
        "query_id": "q03",
        "query": "Что такое амплитуда и как её измерить?",
        "relevant_doc_ids": ["3"],
    },
    {
        "query_id": "q04",
        "query": "Что такое дифракция?",
        "relevant_doc_ids": ["3"],
    },
    {
        "query_id": "q05",
        "query": "Какие частоты у спутников NOAA?",
        "relevant_doc_ids": ["2"],
    },
    {
        "query_id": "q06",
        "query": "Диапазон частот Lime SDR",
        "relevant_doc_ids": ["3"],
    },
]

baseline_eval_k3 = evaluate_benchmark(benchmark_queries, artifacts=artifacts, top_k=3)
display(baseline_eval_k3)
summary_k3 = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "value": [
            baseline_eval_k3["hit_at_k"].mean(),
            baseline_eval_k3["recall@3"].mean(),
            baseline_eval_k3["MRR@3"].mean(),
        ],
    }
)
baseline_eval_k3.to_csv("artifacts/retrieval_eval.csv")
display(summary_k3)


,query_id,query,expected_source,retrieved_sources,hit_at_k,recall@3,MRR@3,rank_of_first_relevant
0,q01,Что такое SDR и зачем он нужен?,0,"0, 1",1,1.0,1.0,1.0
1,q02,Зачем нужен GNU Radio?,1,"0, 3",0,0.0,0.0,NaN
2,q03,Что такое амплитуда и как её измерить?,3,3,1,1.0,1.0,1.0
3,q04,Что такое дифракция?,3,3,1,1.0,1.0,1.0
4,q05,Какие частоты у спутников NOAA?,2,2,1,1.0,1.0,1.0
5,q06,Диапазон частот Lime SDR,3,"1, 0",0,0.0,0.0,NaN


,metric,value
0,mean_hit@3,0.666667
1,mean_recall@3,0.666667
2,mean_MRR@3,0.666667


In [86]:
# Сравниваем несколько конфигураций чанкинга.
chunk_configs = [
    {"chunk_size": 18, "overlap": 4},
    #{"chunk_size": 28, "overlap": 8},
    {"chunk_size": 40, "overlap": 10},
    {"chunk_size": 60, "overlap": 15},
]

chunk_experiments = []

for cfg in chunk_configs:
    exp_artifacts = build_retriever(
        "data/",
        chunk_size=cfg["chunk_size"],
        overlap=cfg["overlap"],
        device=DEVICE,
    )
    eval_df = evaluate_benchmark(benchmark_queries, artifacts=exp_artifacts, top_k=3)

    chunk_experiments.append(
        {
            "chunk_size": cfg["chunk_size"],
            "overlap": cfg["overlap"],
            "num_chunks": len(exp_artifacts.chunks_df),
            "backend_name": exp_artifacts.backend_name,
            "mean_hit@3": eval_df["hit_at_k"].mean(),
            "mean_recall@3": eval_df["recall@3"].mean(),
            "mean_MRR@3": eval_df["MRR@3"].mean(),
        }
    )

chunk_experiments_df = pd.DataFrame(chunk_experiments).sort_values(
    by=["mean_hit@3", "mean_MRR@3", "num_chunks"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(chunk_experiments_df)

,chunk_size,overlap,num_chunks,backend_name,mean_hit@3,mean_recall@3,mean_MRR@3
0,18,4,477,SentenceTransformer: sentence-transformers/par...,0.833333,0.833333,0.750000
1,40,10,223,SentenceTransformer: sentence-transformers/par...,0.666667,0.666667,0.583333
2,60,15,148,SentenceTransformer: sentence-transformers/par...,0.500000,0.500000,0.500000


In [87]:
chunks_df = build_chunks_dataframe("data/", chunk_size=18, overlap=5,extra_data='extra_data/')
chunk_texts = chunks_df["chunk_text"].tolist()

chunk_embeddings = embedder.fit_documents(chunk_texts)
faiss_query = "блоки Throttle Multiply Cons для чего нужны?"
faiss_results_df = search_similar_chunks(faiss_query, top_k=5)

display(Markdown(f"**Запрос:** {faiss_query}"))
display(faiss_results_df)

**Запрос:** блоки Throttle Multiply Cons для чего нужны?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,2,data/Software Defined Radio — как это работает...,47,0.5882,"saved:"", 2*d_info.ret) block += 1 if block > m..."
1,2,3,data/Хакинг бытовых устройств программно-опред...,200,0.5390,забудем задать нашей кнопке имя. Далее вставля...
2,3,3,data/Хакинг бытовых устройств программно-опред...,180,0.5258,"выбирать нужные блоки, перетаскивать их и соед..."
3,4,3,data/Хакинг бытовых устройств программно-опред...,188,0.4892,"в ходе предыдущей атаки. Далее в другом блоке,..."
4,5,2,data/Software Defined Radio — как это работает...,44,0.4845,"blocks to save block, max_blocks = 0, 10 block..."


In [88]:
baseline_eval_k3_new = evaluate_benchmark(benchmark_queries, artifacts=artifacts, top_k=3)
display(baseline_eval_k3_new)
summary_k3 = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "value": [
            baseline_eval_k3["hit_at_k"].mean(),
            baseline_eval_k3["recall@3"].mean(),
            baseline_eval_k3["MRR@3"].mean(),
        ],
    }
)
display(summary_k3)
compare = pd.DataFrame()
compare["query"] = baseline_eval_k3_new["query"]
compare["before_retrieved_sources"] = baseline_eval_k3["retrieved_sources"]
compare["after_retrieved_sources"] = baseline_eval_k3_new["retrieved_sources"]
compare["compare"] = baseline_eval_k3["retrieved_sources"].compare( baseline_eval_k3_new["retrieved_sources"],keep_shape=True)["other"]
compare.to_csv("artifacts/retrieval_before_after_update.csv")


,query_id,query,expected_source,retrieved_sources,hit_at_k,recall@3,MRR@3,rank_of_first_relevant
0,q01,Что такое SDR и зачем он нужен?,0,"0, 1",1,1.0,1.0,1.0
1,q02,Зачем нужен GNU Radio?,1,"0, 3",0,0.0,0.0,NaN
2,q03,Что такое амплитуда и как её измерить?,3,3,1,1.0,1.0,1.0
3,q04,Что такое дифракция?,3,3,1,1.0,1.0,1.0
4,q05,Какие частоты у спутников NOAA?,2,2,1,1.0,1.0,1.0
5,q06,Диапазон частот Lime SDR,3,"1, 0",0,0.0,0.0,NaN


,metric,value
0,mean_hit@3,0.666667
1,mean_recall@3,0.666667
2,mean_MRR@3,0.666667


In [89]:
def build_context_from_retrieval(query: str, artifacts: RetrievalArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: {row['doc_id']} | {row['title']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    # Убираем технические строки источников из ранжирования, но не из общего контекста.
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)



In [90]:


def mini_rag_answer(
    query: str,
    artifacts: RetrievalArtifacts,
    top_k: int = 3,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, artifacts=artifacts, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "question": query,
        "answer": answer,
        "context": context,
        "retrieved_sources": retrieved,
    }
rag_result = mini_rag_answer(
    "блоки Throttle Multiply Cons для чего нужны?",
    artifacts=artifacts,
    top_k=3,
)

display(Markdown(f"### Вопрос: {rag_result['question']}"))
display(Markdown(f"**Ответ:** {rag_result['answer']}"))
display(Markdown("**Источники:**"))
display(rag_result["retrieved_sources"])

### Вопрос: блоки Throttle Multiply Cons для чего нужны?

**Ответ:** Далее вставляем в нашу линию блоков множитель Multiply Const, в качестве значения которого указываем имя кнопки. break Объяснить с Результат на скриншоте: Как можно видеть, мы получаем от устройства блоки данных, размер одного блока составляет 131072 байта, что при частоте дискретизации 250000 дает нам

**Источники:**

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.578131,3,data/Хакинг бытовых устройств программно-опред...,130,забудем задать нашей кнопке имя. Далее вставля...
1,2,0.442327,3,data/Хакинг бытовых устройств программно-опред...,122,"кода, который был получен в ходе предыдущей ат..."
2,3,0.428698,2,data/Software Defined Radio — как это работает...,31,break Объяснить с Результат на скриншоте: Как ...


In [91]:
rag_answers = []
for i in benchmark_queries:
    rag_result = mini_rag_answer(
    i["query"],
    artifacts=artifacts,
    top_k=3,
)
    rag_answers.append(rag_result)
pd.DataFrame(rag_answers).to_csv("artifacts/rag_examples.csv")